[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARODBE/UAX_Fundamento_Ciencia_de_Datos/blob/main/Notebooks/Fase_2_Organizacion_Datos.ipynb)

# Fase 2. Organización y entendimiento de los datos

En esta fase preparo el conjunto de datos que usaré en las siguientes partes. Leo DILIst y una tabla compacta de asociaciones de nombres con estructuras. Calculo los descriptores moleculares de RDKit, reviso la calidad de las coincidencias y guardo los CSV de entrenamiento y prueba. La unidad de análisis será una estructura química identificada por su InChIKey.

Uso DILIst como fuente de etiquetas de lesión hepática. La tabla de estructuras reúne las asociaciones hechas con nombres preferidos de ChEMBL y búsquedas de respaldo en PubChem. Solo incorporo una estructura cuando la identificación es única por InChIKey. El archivo conserva también los nombres sin coincidencia inequívoca para que pueda revisar la cobertura.

In [ ]:
# Instalo RDKit y PubChemPy para calcular propiedades y consultar estructuras.
%pip -q install rdkit pubchempy

# Importo las herramientas que necesito para leer, consultar y preparar los datos.
from pathlib import Path  # Organizo las rutas de salida.
import subprocess  # Clono el proyecto si aún no está en Colab.
import time  # Pauso brevemente entre consultas a las fuentes.
import requests  # Consulto la API de ChEMBL.
import pandas as pd  # Trabajo con tablas.
import numpy as np  # Uso valores numéricos y nulos.
import pubchempy as pcp  # Consulto PubChem cuando ChEMBL no da una coincidencia única.
from rdkit import Chem  # Convierto SMILES en moléculas.
from rdkit.Chem import Descriptors  # Accedo al catálogo de descriptores RDKit.
from IPython.display import display  # Muestro tablas en Colab.

# Clono el repositorio público para guardar los resultados dentro de su carpeta Data.
REPO_DIR = Path('/content/UAX_Fundamento_Ciencia_de_Datos')  # Defino la carpeta local del proyecto.
if not REPO_DIR.exists():  # Compruebo si ya tengo una copia en esta sesión de Colab.
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/PARODBE/UAX_Fundamento_Ciencia_de_Datos.git', str(REPO_DIR)], check=True)  # Descargo el repositorio público.
DATA_DIR = REPO_DIR / 'Data'  # Guardo los archivos generados en Data.
DATA_DIR.mkdir(parents=True, exist_ok=True)  # Creo la carpeta si aún no existe.

# Defino las rutas de los archivos de partida que están en Data.
URL_DILIST = DATA_DIR / 'DILIst_FDA.csv'  # Leo las etiquetas originales del proyecto.
URL_ESTRUCTURAS = DATA_DIR / 'estructuras_nombres.csv'  # Reutilizo el mapa como caché de consultas previas.


In [ ]:
# Cargo DILIst desde el CSV público del proyecto.
dilist_original = pd.read_csv(URL_DILIST)  # Leo las etiquetas originales.
dilist_original.to_csv(DATA_DIR / 'DILIst_FDA.csv', index=False)  # Guardo una copia dentro de Data.

# Compruebo el tamaño y los tipos de datos de la fuente.
print('Filas y columnas de DILIst:', dilist_original.shape)  # Informo las dimensiones.
display(dilist_original.dtypes.astype(str).rename('tipo de dato').to_frame())  # Muestro el tipo de cada columna.
display(dilist_original.head())  # Reviso los primeros registros.

# Describo los campos que utilizaré más adelante.
descripcion = pd.DataFrame({'variable': ['DILIST_ID', 'CompoundName', 'DILIst_Classification', 'Routes_of_Administration'], 'descripción': ['Identificador del registro FDA', 'Nombre del compuesto', 'Etiqueta DILI: 0 negativa y 1 positiva', 'Vía de administración registrada en la fuente']})  # Creo un diccionario breve.
display(descripcion)  # Muestro la descripción de las columnas.

# Aclaro qué representa una fila y la limitación temporal de la fuente.
print('Unidad de análisis: un registro de fármaco de DILIst; la estructura se identificará por InChIKey cuando esté disponible.')  # Describo la unidad.
print('Rango temporal: no disponible; la fuente no contiene una fecha por registro.')  # Informo la limitación temporal.


## 2.3. Obtención de estructuras químicas

A partir de los nombres de DILIst, consulto ChEMBL y uso PubChemPy como respaldo. Acepto una coincidencia solo si conduce a un único InChIKey. Guardo el nombre consultado, el SMILES, la fuente y el estado de la búsqueda en `Data/estructuras_nombres.csv`. Ese archivo también sirve de caché para no repetir las consultas en cada ejecución; las búsquedas para nombres sin una coincidencia resuelta sí se hacen desde el notebook. Si quiero repetir todas las consultas, puedo activar `RECONSULTAR_TODO`.

In [ ]:
# Normalizo nombres para comparar resultados de las dos fuentes.
def normalizar_nombre(nombre):  # Recibo el nombre de un compuesto.
    return ''.join(caracter.lower() for caracter in str(nombre) if caracter.isalnum())  # Quito espacios y signos.

# Busco primero en ChEMBL y acepto solo un InChIKey distinto.
def buscar_chembl(nombre):  # Consulto la API pública de ChEMBL.
    url = 'https://www.ebi.ac.uk/chembl/api/data/molecule/search.json'  # Defino el endpoint de búsqueda.
    respuesta = requests.get(url, params={'q': nombre, 'limit': 20}, timeout=30)  # Busco por nombre.
    respuesta.raise_for_status()  # Aviso si la API devuelve un error HTTP.
    hallazgos = respuesta.json().get('molecules', [])  # Leo los resultados.
    claves = {}  # Agrupo candidatos por InChIKey.
    for item in hallazgos:  # Reviso los resultados devueltos.
        if normalizar_nombre(item.get('pref_name', '')) == normalizar_nombre(nombre):  # Exijo nombre preferido coincidente.
            estructura = item.get('molecule_structures') or {}  # Leo la estructura si la API la incluye.
            clave = estructura.get('standard_inchi_key')  # Extraigo el InChIKey.
            smiles = estructura.get('canonical_smiles')  # Extraigo el SMILES.
            if clave and smiles:  # Conservo registros con estructura completa.
                claves[clave] = {'InChIKey': clave, 'SMILES': smiles, 'source_estructura': 'ChEMBL', 'estado_estructura': 'estructura única'}  # Guardo el candidato.
    if len(claves) == 1:  # Compruebo que haya una sola estructura distinta.
        return list(claves.values())[0]  # Devuelvo la coincidencia inequívoca.
    return None  # Dejo que PubChem intente resolver los casos restantes.

# Si ChEMBL no resuelve el nombre, pruebo la búsqueda de PubChem.
def buscar_pubchem(nombre):  # Consulto PubChemPy por nombre.
    compuestos = pcp.get_compounds(nombre, 'name')  # Recupero los candidatos de PubChem.
    claves = {}  # Agrupo candidatos por InChIKey.
    for compuesto in compuestos:  # Reviso cada candidato.
        clave = getattr(compuesto, 'inchikey', None)  # Leo su identificador químico.
        smiles = getattr(compuesto, 'canonical_smiles', None) or getattr(compuesto, 'isomeric_smiles', None)  # Leo su estructura.
        if clave and smiles:  # Conservo candidatos con ambos campos.
            claves[clave] = {'InChIKey': clave, 'SMILES': smiles, 'source_estructura': 'PubChem', 'estado_estructura': 'estructura única'}  # Guardo el candidato.
    if len(claves) == 1:  # Compruebo que solo haya una estructura distinta.
        return list(claves.values())[0]  # Devuelvo la coincidencia inequívoca.
    return None  # Marco como no resuelto si no hay una única estructura.

# Reutilizo la caché publicada y consulto las estructuras que falten o sean ambiguas.
archivo_estructuras = DATA_DIR / 'estructuras_nombres.csv'  # Defino dónde guardo el mapa generado.
if archivo_estructuras.exists():  # Compruebo si ya lo generé en esta sesión.
    cache = pd.read_csv(archivo_estructuras)  # Reutilizo la caché local.
else:  # Si es una sesión nueva de Colab...
    cache = pd.read_csv(URL_ESTRUCTURAS)  # ...cargo el mapa público para evitar repetir búsquedas resueltas.
RECONSULTAR_TODO = False  # Cambio a True solo si quiero volver a consultar todos los nombres.
cache_por_nombre = {normalizar_nombre(fila['CompoundName']): fila.to_dict() for _, fila in cache.iterrows()}  # Indexo la caché por nombre.
resultados = []  # Creo la lista de asociaciones que guardaré.
nombres = dilist_original['CompoundName'].dropna().astype(str).str.strip().drop_duplicates()  # Tomo los nombres únicos de DILIst.

for numero, nombre in enumerate(nombres, start=1):  # Recorro cada compuesto una vez.
    clave_nombre = normalizar_nombre(nombre)  # Creo la clave de comparación.
    guardado = cache_por_nombre.get(clave_nombre)  # Busco una asociación previa.
    if guardado and guardado.get('InChIKey') and not RECONSULTAR_TODO:  # Reutilizo estructuras ya resueltas.
        registro = guardado.copy()  # Conservo la asociación previa y su procedencia.
    else:  # Si no hay estructura válida, consulto las fuentes.
        try:  # Intento la consulta de ChEMBL.\n            registro = buscar_chembl(nombre)  # Pruebo ChEMBL primero.\n        except Exception:  # Si falla la conexión, pruebo la fuente de respaldo.\n            registro = None  # Dejo que PubChem intente resolverlo.
        if registro is None:  # Si no hay coincidencia única...
            try:  # Intento resolver el caso con PubChem.
                registro = buscar_pubchem(nombre)  # Busco el nombre en PubChem.
            except Exception:  # Registro errores temporales sin detener toda la descarga.
                registro = None  # Dejo el compuesto pendiente de revisión.
        if registro is None:  # Si ninguna fuente devuelve un único InChIKey...
            registro = {'InChIKey': None, 'SMILES': None, 'source_estructura': 'sin coincidencia única', 'estado_estructura': 'sin resolver o ambigua'}  # Marco el resultado.
        time.sleep(0.1)  # Evito enviar consultas seguidas demasiado rápido.
    registro['CompoundName'] = nombre  # Mantengo el nombre exacto de la fuente.
    resultados.append(registro)  # Añado la asociación a la tabla final.
    if numero % 25 == 0:  # Guardo avances para no perder una consulta larga.
        pd.DataFrame(resultados).to_csv(archivo_estructuras, index=False)  # Actualizo el CSV de Data.

# Guardo y reviso el mapa de nombres y estructuras.
estructuras = pd.DataFrame(resultados)  # Convierto las búsquedas en una tabla.
estructuras.to_csv(archivo_estructuras, index=False)  # Guardo el resultado final en Data.
print('Asociaciones guardadas en:', archivo_estructuras)  # Indico la ruta del archivo generado.
display(estructuras['source_estructura'].value_counts(dropna=False).rename('compuestos').to_frame())  # Resumo de qué fuente procede cada estructura.
display(estructuras[estructuras['InChIKey'].isna()].head())  # Muestro los nombres que requieren revisión.


Normalizo los nombres solo para hacer la comparación, pero conservo el nombre original de DILIst en la tabla final. El mapa de estructuras ya incluye una fila por nombre y señala si la estructura quedó identificada, ambigua o sin coincidencia.

In [ ]:
# Renombro la etiqueta para trabajar con un nombre corto y uniforme.
dilist = dilist_original.rename(columns={'DILIst_Classification': 'DILI'}).copy()  # Copio los datos y renombro la clase.
dilist['CompoundName'] = dilist['CompoundName'].astype(str).str.strip()  # Quito espacios sobrantes del nombre.
dilist['DILI'] = pd.to_numeric(dilist['DILI'], errors='coerce')  # Convierto la etiqueta a número.
dilist = dilist.dropna(subset=['CompoundName', 'DILI']).copy()  # Retiro filas sin nombre o clase.

# Preparo claves normalizadas en las dos tablas para unir nombres sin distinguir mayúsculas.
dilist['nombre_clave'] = dilist['CompoundName'].str.upper()  # Normalizo nombres de DILIst.
estructuras['nombre_clave'] = estructuras['CompoundName'].astype(str).str.strip().str.upper()  # Normalizo nombres del mapa.
estructuras = estructuras.drop(columns='CompoundName')  # Quito el nombre duplicado del mapa.
datos = dilist.merge(estructuras, on='nombre_clave', how='left')  # Uno las etiquetas y las estructuras.
datos = datos.drop(columns='nombre_clave')  # Quito la clave auxiliar de trabajo.

# Reviso cuántas filas tienen una estructura inequívoca.
print('Registros DILIst:', len(datos))  # Muestro el total de registros.
print('Con InChIKey:', datos['InChIKey'].notna().sum())  # Cuento las estructuras resueltas.
print('Sin estructura inequívoca:', datos['InChIKey'].isna().sum())  # Cuento los casos pendientes.


Calculo todos los descriptores escalares que trae el catálogo de RDKit (en esta versión, alrededor de 200). Así no reduzco de antemano la información química disponible. Para el EDA describo cinco propiedades fáciles de interpretar —las cuatro de Lipinski y TPSA—; en los modelos selecciono entre el conjunto amplio y retiro variables constantes o muy correlacionadas usando solo entrenamiento. No calculo fingerprints.

In [4]:
# Defino una función que calcula todos los descriptores escalares disponibles en RDKit.
def calcular_descriptores(smiles):  # Recibo una cadena SMILES.
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None  # Creo la molécula si hay SMILES.
    if mol is None:  # Compruebo que RDKit haya leído la estructura.
        return {}  # Devuelvo vacío si no puedo procesarla.
    valores = {}  # Creo un diccionario para las propiedades moleculares.
    for nombre, funcion in Descriptors._descList:  # Recorro el catálogo de descriptores RDKit.
        try:  # Intento calcular cada propiedad.
            valores[nombre] = funcion(mol)  # Guardo el resultado calculado.
        except Exception:  # Evito que un descriptor aislado detenga todo el cálculo.
            valores[nombre] = np.nan  # Marco como ausente el valor que no pude calcular.
    return valores  # Devuelvo todos los descriptores calculados.

# Calculo una fila de propiedades para cada SMILES.
lista_descriptores = [calcular_descriptores(smiles) for smiles in datos['SMILES']]  # Aplico la función a las estructuras.
descriptores = pd.DataFrame(lista_descriptores)  # Convierto los resultados en una tabla.
datos = pd.concat([datos.reset_index(drop=True), descriptores], axis=1)  # Uno los descriptores con las etiquetas.

# Cuento incumplimientos de los cuatro límites de Lipinski.
datos['LipinskiViolations'] = ((datos['MolWt'] >= 500).astype(int) + (datos['MolLogP'] > 5).astype(int) + (datos['NumHDonors'] > 5).astype(int) + (datos['NumHAcceptors'] > 10).astype(int))  # Sumo los cuatro criterios.

# Muestro algunas variables químicas para revisar el cálculo.
columnas_demo = ['CompoundName', 'DILI', 'InChIKey', 'MolWt', 'MolLogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']  # Selecciono columnas legibles.
display(datos[columnas_demo].head())  # Muestro una muestra de los resultados.
print('Descriptores RDKit calculados:', len(descriptores.columns))  # Informo cuántas propiedades se calcularon.


,CompoundName,DILI,InChIKey,MolWt,MolLogP,NumHDonors,NumHAcceptors,TPSA
0,mercaptopurine,1,WFFQYWAAEWLHJC-UHFFFAOYSA-N,170.197,0.19079,2.0,3.0,88.86
1,acetaminophen,1,RZVAJINKPMORJF-UHFFFAOYSA-N,151.165,1.35060,2.0,2.0,49.33
2,azathioprine,1,LMEKQMALGUDUQG-UHFFFAOYSA-N,277.269,1.14580,1.0,7.0,115.42
3,chlorpheniramine,0,SOYKEARSMXGVTM-UHFFFAOYSA-N,274.795,3.81860,0.0,2.0,16.13
4,clofibrate,1,KNHUKKLJHYUCFP-UHFFFAOYSA-N,242.702,3.06050,0.0,3.0,35.53


Descriptores RDKit calculados: 217


## 2.4. Calidad, duplicados y partición

Retiro de los análisis moleculares los registros sin estructura inequívoca. Si la misma InChIKey aparece con etiquetas DILI diferentes, la separo para revisión. Después dejo una fila por InChIKey y creo una partición 80/20. Estratifico por cuantiles de MolLogP y, cuando el tamaño de los grupos lo permite, también por la clase DILI.

In [5]:
# Normalizo la InChIKey y separo los registros con estructura identificada.
datos['InChIKey'] = datos['InChIKey'].astype('string').str.upper()  # Normalizo el identificador químico.
con_etiqueta = datos.dropna(subset=['InChIKey']).copy()  # Conservo moléculas con estructura.

# Identifico claves químicas que aparecen con etiquetas distintas.
clases_por_clave = con_etiqueta.groupby('InChIKey')['DILI'].nunique()  # Cuento etiquetas distintas por estructura.
claves_conflictivas = clases_por_clave[clases_por_clave > 1].index  # Selecciono estructuras con conflicto.
datos_conflicto = con_etiqueta[con_etiqueta['InChIKey'].isin(claves_conflictivas)].copy()  # Guardo esos casos aparte.
datos_limpios = con_etiqueta[~con_etiqueta['InChIKey'].isin(claves_conflictivas)].drop_duplicates('InChIKey').reset_index(drop=True)  # Dejo una fila por estructura válida.

# Busco un número de cuantiles que permita estratificar sin grupos demasiado pequeños.
estratos = None  # Inicio sin una estratificación elegida.
for numero_cuantiles in range(10, 1, -1):  # Pruebo de diez cuantiles a dos.
    cuantiles = pd.qcut(datos_limpios['MolLogP'], q=numero_cuantiles, duplicates='drop')  # Agrupo MolLogP por cuantiles.
    propuesta = datos_limpios['DILI'].astype(int).astype(str) + '_' + cuantiles.astype(str)  # Combino cuantiles y clase.
    if propuesta.value_counts().min() >= 2:  # Compruebo que cada grupo permita dividirse.
        estratos = propuesta  # Guardo la estratificación más detallada disponible.
        break  # Termino la búsqueda.
if estratos is None:  # Compruebo si no fue posible combinar con la clase.
    estratos = cuantiles if cuantiles.value_counts().min() >= 2 else None  # Uso solo cuantiles si es posible.

# Creo la partición fija para que los resultados sean reproducibles.
from sklearn.model_selection import train_test_split  # Importo la función de partición.
train_df, test_df = train_test_split(datos_limpios, test_size=0.20, random_state=42, stratify=estratos)  # Separo 80 % y 20 %.
print('Train:', train_df.shape, '| Test:', test_df.shape)  # Muestro el tamaño de cada parte.
print('Clases en train:', train_df['DILI'].value_counts(normalize=True).round(3).to_dict())  # Reviso la proporción de clases en train.
print('Clases en test:', test_df['DILI'].value_counts(normalize=True).round(3).to_dict())  # Reviso la proporción de clases en test.


Train: (945, 226) | Test: (237, 226)
Clases en train: {1: 0.618, 0: 0.382}
Clases en test: {1: 0.616, 0: 0.384}


## 2.5. Exportación de los CSV

Guardo en la carpeta Data de la copia del proyecto los CSV de partida y los archivos que genero para las fases siguientes.

In [ ]:
# Defino el conjunto de descriptores numéricos calculados, salvo columnas de identificación y resultado.
columnas_excluir = ['DILI', 'DILIST_ID', 'InChIKey', 'SMILES', 'CompoundName', 'Routes_of_Administration', 'chembl_id']  # Evito incluir etiquetas e identificadores como propiedades.
DESCRIPTORES = [columna for columna in descriptores.select_dtypes(include='number').columns if columna not in columnas_excluir]  # Selecciono propiedades numéricas.
DESCRIPTORES = [columna for columna in DESCRIPTORES if columna in datos_limpios.columns]  # Conservo solo columnas presentes en los datos depurados.

# Guardo únicamente las tablas que utilizaré en las fases siguientes.
train_df.to_csv(DATA_DIR / 'dilist_train.csv', index=False)  # Guardo la partición de entrenamiento.
test_df.to_csv(DATA_DIR / 'dilist_test.csv', index=False)  # Guardo la partición de prueba.
datos_limpios.to_csv(DATA_DIR / 'dilist_descriptores.csv', index=False)  # Guardo los datos moleculares depurados.

# Informo dónde quedaron los archivos preparados.
print('He guardado los CSV en:', DATA_DIR)  # Muestro la carpeta de salida.
print('Descriptores numéricos disponibles para los modelos:', len(DESCRIPTORES))  # Muestro el número de propiedades disponibles.


## Interpretación de los datos preparados

Tras retirar nombres sin estructura inequívoca y dejar una fila por InChIKey, trabajo con 1.182 compuestos y 217 descriptores numéricos de RDKit. La partición contiene 945 compuestos en train y 237 en test. La proporción DILI positiva queda prácticamente igual en ambos grupos: 61,8 % en train y 61,6 % en test; la negativa representa 38,2 % y 38,4 %. La estratificación conserva así el balance de clases en las dos particiones. En las fases siguientes calcularé las transformaciones con train y aplicaré esos mismos parámetros a test.
